In [1]:
# ================================
# BDAT05 SESSION 3 — A1 SETUP
# ================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import glob
import os

# 1. Automatically search My Drive for the Kape Tayo workbook
matches = glob.glob(
    "/content/drive/MyDrive/**/KapeTayo_IntegratedCase*.xlsx",
    recursive=True
)

if len(matches) == 0:
    raise FileNotFoundError(
        "KapeTayo_IntegratedCase.xlsx was not found in My Drive. "
        "Make sure you added the professor's folder/file as a shortcut to My Drive."
    )

print("Workbook(s) found:")
for i, path in enumerate(matches):
    print(i, path)

# Use the first matching workbook
FILE = matches[0]

print("\nUsing file:")
print(FILE)

# 2. Check the Excel sheet names
xls = pd.ExcelFile(FILE)

print("\nSheets found:")
print(xls.sheet_names)

# 3. Load the tables needed for Session 3
sales = pd.read_excel(FILE, sheet_name="Sales")
branches = pd.read_excel(FILE, sheet_name="Branches")
calendar = pd.read_excel(FILE, sheet_name="Calendar")

# 4. A1 — inspect Sales
print("\n====================")
print("SALES INFO")
print("====================")

sales.info()

print("\nFirst 10 rows:")
display(sales.head(10))

print("\nRows:", sales.shape[0])
print("Columns:", sales.shape[1])

print("\nMissing values:")
print(sales.isna().sum())

print("\nData types:")
print(sales.dtypes)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workbook(s) found:
0 /content/drive/MyDrive/BDAT04 Kape Tayo Integrated Case (Week 2)/KapeTayo_IntegratedCase.xlsx

Using file:
/content/drive/MyDrive/BDAT04 Kape Tayo Integrated Case (Week 2)/KapeTayo_IntegratedCase.xlsx

Sheets found:
['Sales', 'Customers', 'Branches', 'Inventory', 'Calendar', 'Products', 'Employees']

SALES INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10589 entries, 0 to 10588
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   SaleID         10589 non-null  int64  
 1   Date           10577 non-null  object 
 2   BranchID       10589 non-null  object 
 3   ProductID      10589 non-null  object 
 4   CustomerID     5814 non-null   object 
 5   Quantity       10589 non-null  int64  
 6   UnitPrice      10589 non-null  int64  
 7   Amount         10589 non-nul

,SaleID,Date,BranchID,ProductID,CustomerID,Quantity,UnitPrice,Amount,PaymentMethod
0,109375,2026-05-01,B05,P02,NaN,4,165,660.00,GCash
1,107239,2026-01-18,B01,P06,C0068,1,150,150.00,Cash
2,102371,2025-05-11,B06,P08,C0073,4,95,380.00,Cash
3,104038,2025-08-14,B04,P09,NaN,2,75,150.00,GCash
4,103925,2025-08-09,B02,P02,C0048,1,165,165.00,Cash
5,105769,2025-11-12,B04,P02,NaN,1,165,165.00,Cash
6,104150,2025-08-20,B06,P07,C0090,2,180,360.00,Cash
7,109794,2026-05-22,B01,P12,C0078,1,190,190.00,GCash
8,109394,2026-05-02,B04,P08,C0082,1,95,61.02,Cash
9,105161,2025-10-13,B06,P08,NaN,1,95,95.00,Cash



Rows: 10589
Columns: 9

Missing values:
SaleID              0
Date               12
BranchID            0
ProductID           0
CustomerID       4775
Quantity            0
UnitPrice           0
Amount              0
PaymentMethod       0
dtype: int64

Data types:
SaleID             int64
Date              object
BranchID          object
ProductID         object
CustomerID        object
Quantity           int64
UnitPrice          int64
Amount           float64
PaymentMethod     object
dtype: object


In [2]:
print(sales.dtypes)

sales["Date"] = pd.to_datetime(
    sales["Date"],
    errors="coerce"
)

print("\nDate dtype after conversion:")
print(sales["Date"].dtype)

print("\nMissing/invalid dates after conversion:")
print(sales["Date"].isna().sum())

SaleID             int64
Date              object
BranchID          object
ProductID         object
CustomerID        object
Quantity           int64
UnitPrice          int64
Amount           float64
PaymentMethod     object
dtype: object

Date dtype after conversion:
datetime64[ns]

Missing/invalid dates after conversion:
12


In [3]:
branch_sales = (
    sales.groupby("BranchID")["Amount"]
    .sum()
    .sort_values(ascending=False)
)

print(branch_sales)

BranchID
B01     607298.18
B02     505149.08
B03     421895.03
B05     408700.00
B04     352604.76
B06     214558.75
b01       2750.00
B03       2325.00
B-05      1400.00
B-02      1300.00
b04        975.00
B06        955.00
Name: Amount, dtype: float64


In [4]:
merged = pd.merge(
    sales,
    branches,
    on="BranchID",
    how="inner"
)

print("Before:", len(sales))
print("After:", len(merged))
print("Rows lost:", len(sales) - len(merged))

orphans = set(sales["BranchID"]) - set(branches["BranchID"])
print("Orphan BranchIDs:", orphans)

Before: 10589
After: 10549
Rows lost: 40
Orphan BranchIDs: {'b04', 'B06 ', 'B03 ', 'b01', 'B-05', 'B-02'}


In [5]:
def validate(df, name):
    print(f"=== {name} ===")
    print("rows:", len(df))

    print("\ndtypes:")
    print(df.dtypes)

    print("\nmissing:")
    print(df.isna().sum())

    print("\nduplicate SaleIDs:")
    print(df["SaleID"].duplicated().sum())

    print("\nnegative amounts:")
    print((df["Amount"] < 0).sum())

    orphans = set(df["BranchID"]) - set(branches["BranchID"])
    print("\norphan BranchIDs:")
    print(orphans)

validate(sales, "Sales")

=== Sales ===
rows: 10589

dtypes:
SaleID                    int64
Date             datetime64[ns]
BranchID                 object
ProductID                object
CustomerID               object
Quantity                  int64
UnitPrice                 int64
Amount                  float64
PaymentMethod            object
dtype: object

missing:
SaleID              0
Date               12
BranchID            0
ProductID           0
CustomerID       4775
Quantity            0
UnitPrice           0
Amount              0
PaymentMethod       0
dtype: int64

duplicate SaleIDs:
23

negative amounts:
0

orphan BranchIDs:
{'b04', 'B06 ', 'B03 ', 'b01', 'B-05', 'B-02'}


In [6]:
sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce")
calendar["Date"] = pd.to_datetime(calendar["Date"], errors="coerce")

print("Sales Date:", sales["Date"].dtype)
print("Calendar Date:", calendar["Date"].dtype)

Sales Date: datetime64[ns]
Calendar Date: datetime64[ns]


In [7]:
# Make a clean copy of Sales
sales_clean = sales.drop_duplicates().copy()

# Fix the BranchID formatting problems found in A4
sales_clean["BranchID"] = (
    sales_clean["BranchID"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace("-", "", regex=False)
)

# Join Sales with Calendar
sales_calendar = sales_clean.merge(
    calendar[["Date", "YearMonth"]],
    on="Date",
    how="left"
)

# If a valid date is missing from Calendar, get the month from the sales date
missing_month = (
    sales_calendar["Date"].notna()
    & sales_calendar["YearMonth"].isna()
)

sales_calendar.loc[missing_month, "YearMonth"] = (
    sales_calendar.loc[missing_month, "Date"]
    .dt.to_period("M")
    .astype(str)
)

# Build the branch-month order-count table
p7_slice = (
    sales_calendar
    .dropna(subset=["BranchID", "YearMonth", "SaleID"])
    .groupby(["BranchID", "YearMonth"], as_index=False)
    .agg(OrderCount=("SaleID", "nunique"))
)

display(p7_slice)

print("Rows in table:", len(p7_slice))
print("Number of branches:", p7_slice["BranchID"].nunique())
print("Number of months:", p7_slice["YearMonth"].nunique())

,BranchID,YearMonth,OrderCount
0,B01,2025-01,131
1,B01,2025-02,128
2,B01,2025-03,139
3,B01,2025-04,138
4,B01,2025-05,132
...,...,...,...
103,B06,2026-02,47
104,B06,2026-03,60
105,B06,2026-04,46
106,B06,2026-05,59


Rows in table: 108
Number of branches: 6
Number of months: 18


In [8]:
print(p7_slice.dtypes)

print("\nBranch IDs:")
print(sorted(p7_slice["BranchID"].unique()))

print("\nYearMonth range:")
print(p7_slice["YearMonth"].min(), "to", p7_slice["YearMonth"].max())

print("\nOrderCount range:")
print(p7_slice["OrderCount"].min(), "to", p7_slice["OrderCount"].max())

BranchID      object
YearMonth     object
OrderCount     int64
dtype: object

Branch IDs:
['B01', 'B02', 'B03', 'B04', 'B05', 'B06']

YearMonth range:
2025-01 to 2026-06

OrderCount range:
34 to 182


In [9]:
def validate_p7(df):
    print("=== P7 Branch-Month Slice ===")

    print("\nRows:")
    print(len(df))

    print("\nDtypes:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate branch-month rows:")
    print(df.duplicated(subset=["BranchID", "YearMonth"]).sum())

    print("\nNon-positive OrderCount:")
    print((df["OrderCount"] <= 0).sum())

    print("\nMinimum OrderCount:")
    print(df["OrderCount"].min())

    print("\nMaximum OrderCount:")
    print(df["OrderCount"].max())

validate_p7(p7_slice)

=== P7 Branch-Month Slice ===

Rows:
108

Dtypes:
BranchID      object
YearMonth     object
OrderCount     int64
dtype: object

Missing values:
BranchID      0
YearMonth     0
OrderCount    0
dtype: int64

Duplicate branch-month rows:
0

Non-positive OrderCount:
0

Minimum OrderCount:
34

Maximum OrderCount:
182


The slice uses branch, sales date, and order data to create monthly order counts for each branch. The final table does not include customer names or other personal details. It only shows BranchID, YearMonth, and OrderCount, so individual customers cannot be identified. The raw data should only be accessed by authorized users.  

In [10]:
OUTPUT = "/content/drive/MyDrive/p7_branch_month_order_counts.csv"

p7_slice.to_csv(
    OUTPUT,
    index=False
)

print("Cleaned CSV saved here:")
print(OUTPUT)

Cleaned CSV saved here:
/content/drive/MyDrive/p7_branch_month_order_counts.csv
